# **Experimentation Phase**

## **0_Setup**

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## **1_Load and Inspect the Data**

In [2]:
applications = pd.read_csv('Applications_Compiled.csv')

In [3]:
# formats all floats to two decimal places
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [4]:
applications.head()

,AppName,Category,Rating,ReviewCount,Size_MB,Installs,Type,Price,ContentRating,LastUpdated,MinAndroidVer
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.10,159.00,19.00,"10,000.00",Free,0.00,Everyone,2018-01-07,4.0.3
1,Coloring book moana,ART_AND_DESIGN,3.90,967.00,14.00,"500,000.00",Free,0.00,Everyone,2018-01-15,4.0.3
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.70,"87,510.00",8.70,"5,000,000.00",Free,0.00,Everyone,2018-08-01,4.0.3
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.50,"215,644.00",25.00,"50,000,000.00",Free,0.00,Teen,2018-06-08,4.2
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.30,967.00,2.80,"100,000.00",Free,0.00,Everyone,2018-06-20,4.4


In [5]:
applications.shape

(2199861, 11)

In [6]:
applications.columns

Index(['AppName', 'Category', 'Rating', 'ReviewCount', 'Size_MB', 'Installs',
       'Type', 'Price', 'ContentRating', 'LastUpdated', 'MinAndroidVer'],
      dtype='object')

In [7]:
applications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2199861 entries, 0 to 2199860
Data columns (total 11 columns):
 #   Column         Dtype  
---  ------         -----  
 0   AppName        object 
 1   Category       object 
 2   Rating         float64
 3   ReviewCount    float64
 4   Size_MB        float64
 5   Installs       float64
 6   Type           object 
 7   Price          float64
 8   ContentRating  object 
 9   LastUpdated    object 
 10  MinAndroidVer  object 
dtypes: float64(5), object(6)
memory usage: 184.6+ MB


In [8]:
applications.describe()

,Rating,ReviewCount,Size_MB,Installs,Price
count,"2,199,861.00","2,199,861.00","2,199,861.00","2,199,861.00","2,199,861.00"
mean,2.19,"2,309.22",19.32,"119,007.87",0.11
std,2.11,"99,346.61",23.97,"3,903,970.46",2.86
min,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,4.90,50.00,0.00
50%,2.90,6.00,10.00,500.00,0.00
75%,4.30,40.00,25.00,"5,000.00",0.00
max,5.00,"56,025,424.00","1,020.00","1,000,000,000.00",400.00


In [9]:
applications.isna().sum()

AppName             0
Category            0
Rating              0
ReviewCount         0
Size_MB             0
Installs            0
Type                0
Price               0
ContentRating       0
LastUpdated         0
MinAndroidVer    6323
dtype: int64

In [10]:
# Create a copy of the dataframe
applications_copy = applications.copy()
applications_copy.head()

,AppName,Category,Rating,ReviewCount,Size_MB,Installs,Type,Price,ContentRating,LastUpdated,MinAndroidVer
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.10,159.00,19.00,"10,000.00",Free,0.00,Everyone,2018-01-07,4.0.3
1,Coloring book moana,ART_AND_DESIGN,3.90,967.00,14.00,"500,000.00",Free,0.00,Everyone,2018-01-15,4.0.3
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.70,"87,510.00",8.70,"5,000,000.00",Free,0.00,Everyone,2018-08-01,4.0.3
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.50,"215,644.00",25.00,"50,000,000.00",Free,0.00,Teen,2018-06-08,4.2
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.30,967.00,2.80,"100,000.00",Free,0.00,Everyone,2018-06-20,4.4


In [11]:
# Convert "LastUpdated" column to datetime
applications_copy['LastUpdated'] = pd.to_datetime(applications_copy['LastUpdated'], errors='coerce')

## **2_Handling Missing Values**

In [12]:
# Use 'mode' to impute missing values for "MinAndroidVer"

imputer = SimpleImputer(strategy='most_frequent')

applications_copy[['MinAndroidVer']] = imputer.fit_transform(applications_copy[['MinAndroidVer']])

In [13]:
applications_copy.isna().sum()

AppName          0
Category         0
Rating           0
ReviewCount      0
Size_MB          0
Installs         0
Type             0
Price            0
ContentRating    0
LastUpdated      0
MinAndroidVer    0
dtype: int64

## **3_Feature Engineering 'LastUpdated' Column**

In [14]:
# Reference date set to the newest update in the dataset
max_date = applications_copy['LastUpdated'].max()

# Continuous feature: App update age in days
applications_copy['Days_Since_Update'] = (max_date - applications_copy['LastUpdated']).dt.days

In [15]:
applications_copy['Update_Year'] = applications_copy['LastUpdated'].dt.year
applications_copy['Update_Month'] = applications_copy['LastUpdated'].dt.month
applications_copy['Update_DayOfWeek'] = applications_copy['LastUpdated'].dt.dayofweek  # 0=Monday, 6=Sunday
applications_copy['Is_Weekend'] = applications_copy['Update_DayOfWeek'].isin([5, 6]).astype(int)

In [16]:
applications_copy['Month_Sin'] = np.sin(2 * np.pi * applications_copy['Update_Month'] / 12)
applications_copy['Month_Cos'] = np.cos(2 * np.pi * applications_copy['Update_Month'] / 12)

In [17]:
# Impute continuous elapsed days with median
imputer_date = SimpleImputer(strategy='median')
applications_copy[['Days_Since_Update']] = imputer_date.fit_transform(applications_copy[['Days_Since_Update']])

In [18]:
applications_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2199861 entries, 0 to 2199860
Data columns (total 18 columns):
 #   Column             Dtype         
---  ------             -----         
 0   AppName            object        
 1   Category           object        
 2   Rating             float64       
 3   ReviewCount        float64       
 4   Size_MB            float64       
 5   Installs           float64       
 6   Type               object        
 7   Price              float64       
 8   ContentRating      object        
 9   LastUpdated        datetime64[ns]
 10  MinAndroidVer      object        
 11  Days_Since_Update  float64       
 12  Update_Year        int32         
 13  Update_Month       int32         
 14  Update_DayOfWeek   int32         
 15  Is_Weekend         int64         
 16  Month_Sin          float64       
 17  Month_Cos          float64       
dtypes: datetime64[ns](1), float64(8), int32(3), int64(1), object(5)
memory usage: 276.9+ MB


## **4_Feature Scaling (Numerical Data)**

#### Train/Test Split

In [19]:
num_cols_1 = ['Rating', 'ReviewCount', 'Size_MB', 'Installs', 'Price', 'Days_Since_Update', 'Update_Year']

X_application_copy = applications_copy[num_cols_1]
y_application_copy = applications_copy['Category']

In [20]:
X_train_application_copy, X_test_application_copy, y_train_application_copy, y_test_application_copy = train_test_split(
    X_application_copy, y_application_copy, test_size=0.2, random_state=42
)

In [21]:
print(X_train_application_copy.shape)
print(X_test_application_copy.shape)

(1759888, 7)
(439973, 7)


#### Scaling

In [22]:
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train_application_copy) # TRAINING DATA ONLY
X_test_scale = scaler.transform(X_test_application_copy) # TEST DATA

# **Pipeline + ColumnTransformer**

### 'Category' as the Dependent Variable

***What categories of apps have similar profiles?***

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Select features for clustering
# Category is intentionally excluded.
# Rating is included as a characteristic of each app.

num_cols_1 = ['Rating', 'ReviewCount', 'Size_MB', 'Installs', 'Price', 'Days_Since_Update', 'Update_Year']
cat_cols = ['ContentRating', 'Type', 'MinAndroidVer']
passthrough_cols = ['Is_Weekend', 'Month_Sin', 'Month_Cos']

feature_cols = num_cols_1 + cat_cols + passthrough_cols

# Create the feature matrix
X = applications_copy[feature_cols].copy()

# Inspect selected features
print("Feature matrix shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

# Pipeline
# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_cols_1),
        ('cat', cat_pipeline, cat_cols),
        ('pass', 'passthrough', passthrough_cols)
    ],
    remainder='drop'
)

# Transform the Features
# Fit the preprocessing pipeline and transform all app records
X_cluster = preprocessor.fit_transform(X)

# Get the names of the transformed features
feature_names = preprocessor.get_feature_names_out()

X_cluster_df = pd.DataFrame(X_cluster, columns=feature_names, index=X.index)

print("Original feature matrix:", X.shape)
print("Transformed feature matrix:", X_cluster_df.shape)

Feature matrix shape: (2199861, 13)

Missing values:
Rating               0
ReviewCount          0
Size_MB              0
Installs             0
Price                0
Days_Since_Update    0
Update_Year          0
ContentRating        0
Type                 0
MinAndroidVer        0
Is_Weekend           0
Month_Sin            0
Month_Cos            0
dtype: int64
Original feature matrix: (2199861, 13)
Transformed feature matrix: (2199861, 45)
